# MobileNetV2 Flood Detector Training

This notebook fine-tunes MobileNetV2 on the **Kaggle Flood Area Segmentation** dataset.

**Steps:**
1. Download dataset from Kaggle
2. Prepare data loaders
3. Fine-tune MobileNetV2 (transfer learning)
4. Evaluate and export model

**Runtime:** Use GPU (Runtime > Change runtime type > GPU)

## 1. Setup & Install Dependencies

In [ ]:
# Install kaggle API
!pip install -q kaggle

# Upload your kaggle.json API key
# Get it from: https://www.kaggle.com/settings -> API -> Create New Token
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 2. Download Kaggle Dataset

In [ ]:
# Download the Flood Area Segmentation dataset
!kaggle datasets download -d faizalkarim/flood-area-segmentation
!unzip -q flood-area-segmentation.zip -d data/

import os
print("\nDataset contents:")
for root, dirs, files in os.walk("data"):
    level = root.replace("data", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files[:5]:
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files)-5} more files")

## 3. Prepare Data Loaders

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
from pathlib import Path
import random

# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class FloodDataset(Dataset):
    """
    Loads images and masks.
    Label = 1 (flood) if mask has >10% white pixels, else 0 (no flood).
    """
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        
        # Get all images
        self.images = sorted(list(self.image_dir.glob("*.jpg")) + 
                            list(self.image_dir.glob("*.png")))
        print(f"Found {len(self.images)} images")
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        
        # Load image
        image = Image.open(img_path).convert("RGB")
        
        # Try to find corresponding mask
        mask_path = self.mask_dir / img_path.name
        if not mask_path.exists():
            # Try different extensions
            for ext in [".png", ".jpg", ".jpeg"]:
                alt_path = self.mask_dir / (img_path.stem + ext)
                if alt_path.exists():
                    mask_path = alt_path
                    break
        
        # Derive label from mask
        if mask_path.exists():
            mask = Image.open(mask_path).convert("L")
            mask_np = np.array(mask)
            water_ratio = (mask_np > 128).mean()
            label = 1 if water_ratio > 0.1 else 0
        else:
            # If no mask, assume flood (since it's a flood dataset)
            label = 1
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Find dataset directories (adjust based on actual structure)
data_root = Path("data")

# Common structures:
possible_image_dirs = [
    data_root / "Image",
    data_root / "images",
    data_root / "train" / "images",
    data_root / "Flood Area Segmentation Dataset" / "train_images",
]

possible_mask_dirs = [
    data_root / "Mask",
    data_root / "masks",
    data_root / "train" / "masks",
    data_root / "Flood Area Segmentation Dataset" / "train_masks",
]

image_dir = None
mask_dir = None

for d in possible_image_dirs:
    if d.exists():
        image_dir = d
        print(f"Found images at: {d}")
        break

for d in possible_mask_dirs:
    if d.exists():
        mask_dir = d
        print(f"Found masks at: {d}")
        break

if not image_dir:
    print("Could not find image directory! Listing data folder:")
    !find data -type d
    raise ValueError("Please set image_dir manually based on output above")

In [ ]:
# Create dataset
full_dataset = FloodDataset(image_dir, mask_dir, transform=train_transform)

# Split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size]
)

# Override transforms for validation
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

## 4. Build MobileNetV2 Model

In [ ]:
# Load pretrained MobileNetV2
model = models.mobilenet_v2(pretrained=True)

# Replace classifier head for binary classification
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 1),  # Single output for binary
    nn.Sigmoid()  # Output probability 0-1
)

model = model.to(device)

# Freeze early layers (optional - for faster training)
for param in model.features[:10].parameters():
    param.requires_grad = False

print(f"Model ready. Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Training Loop

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)
            
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            predicted = (outputs > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

In [ ]:
# Training
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2%}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2%}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "mobilenetv2_flood_best.pth")
        print(f"  -> New best model saved!")
    print()

print("-" * 60)
print(f"Training complete! Best validation accuracy: {best_val_acc:.2%}")

## 6. Visualize Training

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

## 7. Export Model

In [ ]:
# Load best weights
model.load_state_dict(torch.load("mobilenetv2_flood_best.pth"))
model.eval()

# Save full model state
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'architecture': 'mobilenet_v2',
        'num_classes': 1,
        'input_size': IMG_SIZE,
        'best_val_acc': best_val_acc
    }
}, "mobilenetv2_flood_final.pth")

print("Model exported: mobilenetv2_flood_final.pth")
print(f"Best validation accuracy: {best_val_acc:.2%}")

In [ ]:
# Download the trained model
from google.colab import files
files.download('mobilenetv2_flood_final.pth')
files.download('training_history.png')

## 8. Integration Instructions

After downloading `mobilenetv2_flood_final.pth`:

1. Place it in `models/` directory of your project
2. Update `image_classifier.py` to load this model:

```python
# In __init__:
checkpoint = torch.load("models/mobilenetv2_flood_final.pth", map_location='cpu')
self.model.load_state_dict(checkpoint['model_state_dict'])
```

3. The model now outputs a direct flood probability (0-1) instead of ImageNet classes